# Day 4: Social Science Applications

**LLMs for Social Science** | Oxford Spring School

| Day | Theme | Status |
|-----|-------|--------|
| 1 | From Embeddings to Transformers | Done |
| 2 | From Models to Tools | Done |
| 3 | Deploying for Research | Done |
| **4** | **Social Science Applications** | **Today** |
| 5 | Agentic Workflows | Next |

## Why this matters

Over the first three days you learned how models work, how to prompt and validate them, and how to fine-tune when prompting is not enough. Today you **apply everything to real social science research tasks** beyond classification.

## Today's outline

- **Section 1:** Information Extraction (~30 min)
- *Break (~15 min)*
- **Section 2:** Retrieval-Augmented Generation (~60 min)
- **Section 3:** LLMs as Simulated Agents (~30 min)
- **Closing** (~10 min)

## Setup

In [ ]:
#@title Install dependencies
!pip install -q transformers accelerate bitsandbytes
!pip install -q sentence-transformers faiss-cpu
!pip install -q torch pandas numpy scikit-learn tqdm

import torch
import pandas as pd
import numpy as np
import json
from tqdm import tqdm

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")
if device == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Memory: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")


In [ ]:
#@title Load the language model: Qwen2.5-7B-Instruct (4-bit)

from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

model_name = "Qwen/Qwen2.5-7B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(model_name, padding_side='left')
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map="auto",
)

print(f"Model loaded: {model_name} (4-bit)")
print(f"GPU memory used: {torch.cuda.memory_allocated() / 1e9:.1f} GB")


In [ ]:
#@title Generation helper

def generate_response(user_message, max_new_tokens=300, temperature=0.0):
    """Generate a response from the instruct model."""
    messages = [{"role": "user", "content": user_message}]
    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    with torch.no_grad():
        output = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=(temperature > 0),
            temperature=temperature if temperature > 0 else None,
        )

    new_tokens = output[0][inputs['input_ids'].shape[1]:]
    return tokenizer.decode(new_tokens, skip_special_tokens=True).strip()


In [ ]:
#@title UK manifesto corpus

# Excerpts from UK party manifestos (Labour, Conservative, Liberal Democrats)
# across the 2017, 2019, and 2024 general elections, covering key policy areas.
# Each excerpt is a self-contained paragraph suitable for retrieval.

MANIFESTO_CORPUS = [
    # --- ECONOMY ---
    {
        "id": "lab_2017_econ_1",
        "party": "Labour",
        "year": 2017,
        "topic": "economy",
        "text": "Labour will invest in infrastructure and skills to build an economy that works for everyone, not just those at the top. We will establish a National Investment Bank capitalized with 250 billion pounds to channel investment to every region and nation of the UK, supporting one million new jobs."
    },
    {
        "id": "con_2017_econ_1",
        "party": "Conservative",
        "year": 2017,
        "topic": "economy",
        "text": "We will continue to reduce the deficit so that we are no longer spending more than we earn. We will keep taxes as low as possible, maintain our commitment to the triple lock on pensions, and ensure that Britain remains one of the most competitive economies in the world."
    },
    {
        "id": "lib_2017_econ_1",
        "party": "Liberal Democrats",
        "year": 2017,
        "topic": "economy",
        "text": "We will provide a stable, sustainable economic framework that promotes investment and innovation. This means remaining in the single market, reforming business rates to support high street shops, and introducing a one penny rise on income tax to fund the NHS and social care."
    },
    {
        "id": "lab_2019_econ_1",
        "party": "Labour",
        "year": 2019,
        "topic": "economy",
        "text": "Labour will launch the largest investment programme in living memory, investing over 400 billion pounds in infrastructure and industry. This Green Industrial Revolution will create one million climate jobs and bring major industries into public ownership, including rail, mail, water, and energy."
    },
    {
        "id": "con_2019_econ_1",
        "party": "Conservative",
        "year": 2019,
        "topic": "economy",
        "text": "Our economic plan is built on getting Brexit done and unleashing the potential of the whole United Kingdom. We will not raise the rate of income tax, VAT, or National Insurance. We will invest in infrastructure with a 100 billion pound programme and level up every part of the country."
    },
    {
        "id": "lib_2019_econ_1",
        "party": "Liberal Democrats",
        "year": 2019,
        "topic": "economy",
        "text": "Stopping Brexit is the biggest boost we can give the economy. On top of that, we will invest in infrastructure, skills, and research to build a stronger, greener economy. We will introduce a frequent-flyer levy and use the proceeds to fund green transport alternatives."
    },
    {
        "id": "lab_2024_econ_1",
        "party": "Labour",
        "year": 2024,
        "topic": "economy",
        "text": "Labour will deliver economic stability as the foundation for growth. We will maintain strict fiscal rules: the current budget must move into balance and debt must be falling as a share of the economy. We will create a National Wealth Fund to invest alongside the private sector in the industries of the future."
    },
    {
        "id": "con_2024_econ_1",
        "party": "Conservative",
        "year": 2024,
        "topic": "economy",
        "text": "Our plan is to cut taxes for workers and grow the economy. We will abolish the main rate of employee National Insurance entirely by 2030, saving the average worker over 1,300 pounds per year. We will maintain our commitment to fiscal responsibility while reducing the tax burden."
    },

    # --- HEALTHCARE ---
    {
        "id": "lab_2017_health_1",
        "party": "Labour",
        "year": 2017,
        "topic": "healthcare",
        "text": "Labour will guarantee the funding the NHS needs, delivering an extra 30 billion pounds over the next parliament. We will end the crisis in social care with a new National Care Service and ensure that no one has to sell their home to pay for care."
    },
    {
        "id": "con_2019_health_1",
        "party": "Conservative",
        "year": 2019,
        "topic": "healthcare",
        "text": "We will build and fund 40 new hospitals over the next ten years and deliver 50,000 more nurses. We will maintain the triple lock on pensions, introduce an integrated health and social care plan, and ensure that nobody has to sell their home to pay for social care."
    },
    {
        "id": "lab_2024_health_1",
        "party": "Labour",
        "year": 2024,
        "topic": "healthcare",
        "text": "Labour will cut NHS waiting times with 40,000 more evening and weekend appointments each week, paid for by cracking down on tax avoidance and non-dom status. We will recruit thousands more GPs and reform the system to move from hospital to community care, making the NHS fit for the future."
    },
    {
        "id": "lib_2024_health_1",
        "party": "Liberal Democrats",
        "year": 2024,
        "topic": "healthcare",
        "text": "We will give everyone the right to see a GP within seven days, or within 24 hours if urgent. We will fix the social care crisis by introducing free personal care, funded by a dedicated Health and Social Care Tax. We will invest in mental health services to ensure parity of esteem with physical health."
    },

    # --- IMMIGRATION ---
    {
        "id": "con_2017_imm_1",
        "party": "Conservative",
        "year": 2017,
        "topic": "immigration",
        "text": "We will reduce immigration to sustainable levels, bringing net migration down to the tens of thousands. We will remain signatories to the European Convention on Human Rights but will consider further reform. Leaving the EU means we can control our borders and decide who comes to Britain."
    },
    {
        "id": "lab_2019_imm_1",
        "party": "Labour",
        "year": 2019,
        "topic": "immigration",
        "text": "Labour will develop a humane immigration policy that recognises the contribution of migrants to our society and economy. We will end indefinite detention, close Yarl's Wood and Brook House detention centres, and ensure that family reunion rights are maintained after Brexit."
    },
    {
        "id": "con_2024_imm_1",
        "party": "Conservative",
        "year": 2024,
        "topic": "immigration",
        "text": "We will cap the overall number of visas issued each year, set by Parliament, so that the British people have democratic control over immigration levels. We will maintain our Rwanda partnership to deter illegal entry and process asylum claims offshore."
    },
    {
        "id": "lab_2024_imm_1",
        "party": "Labour",
        "year": 2024,
        "topic": "immigration",
        "text": "Labour will reduce net migration with a properly managed system. We will reform the points-based immigration system by linking visa issuance to training plans so that employers invest in British workers. We will scrap the Rwanda scheme and instead use the money to fund a new Border Security Command."
    },

    # --- CLIMATE / ENVIRONMENT ---
    {
        "id": "lab_2019_climate_1",
        "party": "Labour",
        "year": 2019,
        "topic": "climate",
        "text": "Labour will launch a Green New Deal to achieve net zero carbon emissions by 2030. We will invest in renewable energy, insulate millions of homes, and create hundreds of thousands of green jobs. Energy and water will be brought into public ownership to deliver clean energy for all."
    },
    {
        "id": "con_2019_climate_1",
        "party": "Conservative",
        "year": 2019,
        "topic": "climate",
        "text": "We will reach net zero greenhouse gas emissions by 2050. We will invest 9.2 billion pounds in the energy efficiency of homes, schools, and hospitals and lead the global fight against climate change. We will establish new national parks and plant 30 million trees."
    },
    {
        "id": "lib_2019_climate_1",
        "party": "Liberal Democrats",
        "year": 2019,
        "topic": "climate",
        "text": "We will reach net zero greenhouse gas emissions by 2045, five years ahead of the current target. We will plant 60 million trees a year, ban fracking, and generate 80 percent of electricity from renewables by 2030. A green future is a Liberal Democrat future."
    },
    {
        "id": "lab_2024_climate_1",
        "party": "Labour",
        "year": 2024,
        "topic": "climate",
        "text": "Labour will set up Great British Energy, a new publicly owned clean energy company headquartered in Scotland. We will double onshore wind, triple solar power, and quadruple offshore wind by 2030. This will cut energy bills, create jobs, and make Britain energy independent."
    },

    # --- EDUCATION ---
    {
        "id": "lab_2017_edu_1",
        "party": "Labour",
        "year": 2017,
        "topic": "education",
        "text": "Labour will abolish university tuition fees and restore maintenance grants for the poorest students. We will introduce a National Education Service for lifelong learning, reduce class sizes for five, six, and seven year olds, and ensure every school has the funding it needs."
    },
    {
        "id": "con_2017_edu_1",
        "party": "Conservative",
        "year": 2017,
        "topic": "education",
        "text": "We will build on our reforms to create more good school places, open new free schools and university technical colleges, and lift the ban on new grammar schools. We will ensure that every child has the opportunity to attend a good or outstanding school, regardless of their background."
    },
    {
        "id": "lib_2024_edu_1",
        "party": "Liberal Democrats",
        "year": 2024,
        "topic": "education",
        "text": "We will invest in education from early years through university. We will extend free school meals to all primary school children, recruit more teachers with better pay and conditions, and triple the early years pupil premium to give every child the best start in life."
    },

    # --- HOUSING ---
    {
        "id": "lab_2024_housing_1",
        "party": "Labour",
        "year": 2024,
        "topic": "housing",
        "text": "Labour will build 1.5 million new homes over the next parliament by reforming planning rules, hiring hundreds of new planning officers, and building a new generation of council and social housing. We will introduce a permanent mortgage guarantee scheme and reform the private rental market."
    },
    {
        "id": "con_2024_housing_1",
        "party": "Conservative",
        "year": 2024,
        "topic": "housing",
        "text": "We will deliver 1.6 million new homes by fast-tracking building on brownfield sites and investing in urban regeneration. We will protect the green belt, abolish no-fault evictions, and help young people onto the housing ladder with a new Help to Buy scheme."
    },
    {
        "id": "lib_2024_housing_1",
        "party": "Liberal Democrats",
        "year": 2024,
        "topic": "housing",
        "text": "We will build 380,000 new homes a year, including 150,000 social homes, by hiring more planning officers, releasing land held by developers, and establishing a new Housing and Communities Agency. We will end the scandal of homelessness with a comprehensive strategy and proper funding."
    },

    # --- DEFENCE / FOREIGN POLICY ---
    {
        "id": "con_2024_defence_1",
        "party": "Conservative",
        "year": 2024,
        "topic": "defence",
        "text": "We will increase defence spending to 2.5 percent of GDP by 2030, maintain our nuclear deterrent, and continue to lead NATO support for Ukraine. Britain will remain a leading force for global security, with a defence industrial strategy to invest in British manufacturing."
    },
    {
        "id": "lab_2024_defence_1",
        "party": "Labour",
        "year": 2024,
        "topic": "defence",
        "text": "Labour will maintain the UK's nuclear deterrent and commit to NATO. We will set out a path to spending 2.5 percent of GDP on defence and conduct a Strategic Defence Review in the first year of government. We will continue to stand with Ukraine for as long as it takes."
    },
]

# Convert to DataFrame for easy manipulation
corpus_df = pd.DataFrame(MANIFESTO_CORPUS)
print(f"Manifesto corpus: {len(corpus_df)} excerpts")
print(f"Parties: {corpus_df['party'].unique().tolist()}")
print(f"Years: {sorted(corpus_df['year'].unique().tolist())}")
print(f"Topics: {corpus_df['topic'].unique().tolist()}")
print(f"\nSample excerpt:")
print(f"  [{corpus_df.iloc[0]['party']} {corpus_df.iloc[0]['year']}, {corpus_df.iloc[0]['topic']}]")
print(f"  {corpus_df.iloc[0]['text'][:150]}...")


---

# Section 1: Information Extraction

In Days 2 and 3, you classified text: the output was one label per document. Many research tasks need more: extracting **structured information** from text. What policy is being proposed? Who benefits? What justification is given?

This section applies the structured output skills from Day 2 to a harder problem, and introduces a new concern: **faithfulness**. When the model extracts or summarizes, does it stick to what the text actually says, or does it add, omit, or distort?

### Exercise 1: Extract and Verify

This exercise has two parts. First, write a prompt that extracts structured information from a manifesto paragraph. Then, evaluate whether the extraction is *faithful* to the source.

**Part A:** Write an extraction prompt that returns JSON with these fields:
- `policy`: what is being proposed (one sentence)
- `position`: the ideological direction (e.g., "increase spending", "cut taxes")
- `target_group`: who benefits or is affected
- `justification`: why the party says this is needed

**Part B:** For each extraction, check every claim against the source text. Mark each as:
- **F** (Faithful): directly supported by the text
- **H** (Hallucination): not in the original
- **D** (Distortion): emphasis or framing shifted

In [ ]:
# Part A: Write your extraction prompt. Use {text} as placeholder.
extraction_prompt = ""  # YOUR CODE HERE

# Run on 5 manifesto excerpts
if extraction_prompt:
    test_excerpts = corpus_df.sample(5, random_state=42)
    for _, row in test_excerpts.iterrows():
        print(f"[{row['party']} {row['year']}, {row['topic']}]")
        print(f"SOURCE: {row['text'][:200]}")
        response = generate_response(extraction_prompt.format(text=row['text']), max_new_tokens=200)
        print(f"EXTRACTION: {response[:300]}")
        print()
        # Part B: For each extraction above, annotate:
        # Which claims are F (faithful), H (hallucination), or D (distortion)?

The most dangerous failure mode for research is **hallucination**: the model adds plausible-sounding claims not present in the source. Unlike classification errors (which are easy to count), hallucinated extractions look authoritative and require manual verification against the source text.

**Summarization** has the same failure modes: hallucination (adds facts), omission (drops key points), and distortion (shifts emphasis). Always verify extracted or summarized claims against the source.

---

*Take a 15-minute break. When we come back: how to work with corpora too large for any context window.*

---

# Section 2: Retrieval-Augmented Generation (RAG)

## The problem

Your research corpus has hundreds of pages of text across parties and elections. No model can process all of that at once. Even models with 128k-token context windows struggle with full collections, and performance degrades with very long contexts.

## The solution: RAG

Retrieval-Augmented Generation works in four steps:

1. **Chunk:** Split your corpus into small, self-contained pieces (paragraphs, sections)
2. **Embed:** Convert each chunk into a vector using an embedding model (callback to Day 1: this is the same idea as word embeddings, applied to full passages)
3. **Index:** Store the vectors in a searchable index for fast retrieval
4. **Retrieve + Generate:** When a user asks a question, embed the question, find the most similar chunks, and pass them to the language model as context

The model never sees the full corpus. It only sees the most relevant pieces, selected by semantic similarity.

### Step 1: Chunking

Our corpus is already chunked (each manifesto excerpt is a paragraph). In practice, you would need to decide how to split your documents. Chunk size is a design choice:

- **Too small** (individual sentences): loses context, retrieval may return fragments
- **Too large** (full pages): dilutes the relevant signal with irrelevant text, uses more tokens
- **Right size** (paragraphs, ~100-300 words): self-contained units of meaning

For manifesto text, paragraph-level chunking works well. For other corpora (legal documents, parliamentary debates), you might chunk by section, speech turn, or a sliding window.


In [ ]:
# Our corpus is already paragraph-level chunks
print(f"Corpus: {len(corpus_df)} chunks")
print(f"Average chunk length: {corpus_df['text'].str.split().str.len().mean():.0f} words")
print(f"Range: {corpus_df['text'].str.split().str.len().min()}-{corpus_df['text'].str.split().str.len().max()} words")


### Step 2: Embed

We use a **sentence embedding model** to convert each chunk into a dense vector. Unlike the word embeddings from Day 1 (one vector per word), sentence embeddings capture the meaning of an entire passage in a single vector.

We use `all-MiniLM-L6-v2` from the `sentence-transformers` library: small (90MB), fast, and effective for retrieval tasks. It runs on CPU, so it does not compete with our language model for GPU memory.


In [ ]:
#@title Embed all chunks

from sentence_transformers import SentenceTransformer

# Load embedding model (runs on CPU, ~90MB)
embed_model = SentenceTransformer('all-MiniLM-L6-v2', device='cpu')

# Embed all chunks
chunk_texts = corpus_df['text'].tolist()
chunk_embeddings = embed_model.encode(chunk_texts, show_progress_bar=True, convert_to_numpy=True)

print(f"Embedded {len(chunk_embeddings)} chunks")
print(f"Embedding dimension: {chunk_embeddings.shape[1]}")
print(f"Each chunk is now a vector of {chunk_embeddings.shape[1]} numbers")


### Step 3: Index with FAISS

[FAISS](https://github.com/facebookresearch/faiss) (Facebook AI Similarity Search) is a library for fast nearest-neighbor search over dense vectors. For 28 chunks, numpy cosine similarity would work fine. But FAISS scales to millions of vectors, which is what you need for real research corpora.

The index stores all chunk embeddings and lets you quickly find the most similar ones to any query vector.


In [ ]:
#@title Build FAISS index

import faiss

# Normalize embeddings for cosine similarity
# (FAISS IndexFlatIP with normalized vectors = cosine similarity)
faiss.normalize_L2(chunk_embeddings)

# Build the index
embedding_dim = chunk_embeddings.shape[1]
index = faiss.IndexFlatIP(embedding_dim)  # Inner Product (= cosine after normalization)
index.add(chunk_embeddings)

print(f"FAISS index built: {index.ntotal} vectors, dimension {embedding_dim}")
print(f"Index type: Flat (exact search). For larger corpora, use IndexIVFFlat or HNSW.")


In [ ]:
#@title Retrieval function

def retrieve(query, k=3):
    """
    Retrieve the top-k most relevant chunks for a given query.

    Args:
        query: A natural language question or search string.
        k: Number of chunks to retrieve.

    Returns:
        List of dicts with 'text', 'party', 'year', 'topic', and 'score'.
    """
    # Embed the query
    query_vec = embed_model.encode([query], convert_to_numpy=True)
    faiss.normalize_L2(query_vec)

    # Search the index
    scores, indices = index.search(query_vec, k)

    results = []
    for score, idx in zip(scores[0], indices[0]):
        row = corpus_df.iloc[idx]
        results.append({
            'text': row['text'],
            'party': row['party'],
            'year': row['year'],
            'topic': row['topic'],
            'score': float(score),
            'id': row['id'],
        })
    return results


### Exercise 2: Test Retrieval

The core operation: given a research question, find the most relevant chunks. Your task: write research questions and evaluate whether the retriever finds the right passages.

Pay attention to: does it find the right party? The right topic? Try synonyms ("immigration" vs "border control") and see what changes.

In [ ]:
# Write research questions and evaluate retrieval quality
queries = [
    "What is Labour's position on immigration?",
    "Which parties support increasing defence spending?",
    "What do the manifestos say about housing?",
    # Add your own:
    # "",
]

for query in queries:
    if not query: continue
    print(f"QUERY: {query}")
    results = retrieve(query, k=3)
    for i, r in enumerate(results):
        print(f"  [{i+1}] ({r['party']} {r['year']}, {r['topic']}) score={r['score']:.3f}")
        print(f"      {r['text'][:100]}...")
    print()

In [ ]:
#@title RAG generation function

def rag_answer(query, k=3):
    """
    Answer a question using the RAG pipeline:
    retrieve relevant chunks, then generate an answer grounded in them.
    """
    # Step 1: Retrieve
    results = retrieve(query, k=k)

    # Step 2: Format context
    context_parts = []
    for r in results:
        context_parts.append(f"[{r['party']} {r['year']}, {r['topic']}]: {r['text']}")
    context = "\n\n".join(context_parts)

    # Step 3: Generate with context
    prompt = (
        "You are a research assistant analyzing UK party manifestos. "
        "Answer the question below using ONLY the provided manifesto excerpts. "
        "Cite which party and year you are drawing from. "
        "If the excerpts do not contain enough information to answer, say so.\n\n"
        f"MANIFESTO EXCERPTS:\n{context}\n\n"
        f"QUESTION: {query}\n\n"
        "ANSWER:"
    )

    answer = generate_response(prompt, max_new_tokens=300)
    return answer, results


### Exercise 3: RAG vs. No-RAG

Now combine retrieval with generation. Ask a research question, compare the RAG answer to the model's answer *without* retrieval, and evaluate which is more faithful.

This is the aha moment: without retrieval, the model draws on training data (which may be wrong, outdated, or hallucinated). With retrieval, it is grounded in your actual corpus.

In [ ]:
query = "What is Labour's position on public ownership?"  # change this

# RAG answer (grounded in retrieved passages)
rag_response, retrieved = rag_answer(query, k=3)
print(f"QUERY: {query}")
print(f"\nRAG ANSWER (grounded in {len(retrieved)} passages):")
print(rag_response)
print()
print("Retrieved passages:")
for r in retrieved:
    print(f"  [{r['party']} {r['year']}] {r['text'][:80]}...")

# No-RAG answer (model's own knowledge)
no_rag_prompt = (
    "You are a research assistant analyzing UK party manifestos. "
    f"Answer this question: {query}"
)
no_rag_response = generate_response(no_rag_prompt, max_new_tokens=300)
print(f"\nNO-RAG ANSWER (model's own knowledge):")
print(no_rag_response)

# Which is more faithful to what the manifestos actually say?
# Does the no-RAG answer include claims not in any manifesto?

### Exercise 4: Cross-Party Comparison via RAG

A realistic research application: use RAG to systematically compare party positions on the same issue. Notice whether the retriever correctly matches each party to its own text.

In [ ]:
topic = "climate change and energy"  # change this
parties = ["Labour", "Conservative", "Liberal Democrats"]

print(f"CROSS-PARTY COMPARISON: {topic}")
for party in parties:
    query = f"What is {party}'s position on {topic}?"
    answer, retrieved = rag_answer(query, k=2)
    party_chunks = [r for r in retrieved if r['party'] == party]
    other_chunks = [r for r in retrieved if r['party'] != party]
    print(f"\n{party.upper()}")
    print(f"  Retrieved: {len(party_chunks)} from {party}, {len(other_chunks)} from others")
    print(f"  Answer: {answer[:250]}")
    if other_chunks:
        print(f"  [Note: also pulled from: {[r['party'] for r in other_chunks]}]")
    print()

**Section takeaway:** RAG transforms the model from an unreliable knowledge source into a retriever + synthesizer grounded in your actual data. Design choices at every step (chunk size, embedding model, k, metadata filters) affect quality. For research, always verify that retrieved passages are relevant and that the generated answer is faithful to them.

---

# Section 3: LLMs as Simulated Agents

## The homo silicus paradigm

Can language models stand in for human survey respondents? The idea: prompt an LLM with a demographic profile ("55-year-old conservative woman, university-educated, lives in a rural area") and ask it to respond to survey questions as that person would.

This is **silicon sampling** (Argyle et al., 2023): using LLMs to simulate human populations. The promise is real: instant, cheap, scalable survey data. And so are the problems.

In this section, you will use **real survey respondent data** from the World Values Survey to test this. Instead of making up personas, you will take an actual respondent's demographic profile, hold out one of their survey responses, and see if the model can predict it.

In [ ]:
#@title WVS metadata and survey data

# Curated subset of World Values Survey metadata
WVS_METADATA = {
  "profile_features": {
    "Q260": {
      "description": "Sex",
      "question": "How do you identify your sex, male or female?",
      "topic_tag": null,
      "original_section": "demographics",
      "values": {
        "1": "Male",
        "2": "Female",
        "-2": "No answer",
        "-4": "Not asked",
        "-5": "Missing"
      }
    },
    "Q262": {
      "description": "Age",
      "question": "How old are you?",
      "topic_tag": null,
      "original_section": "demographics",
      "values": {
        "-1": "Don't know",
        "-2": "No answer",
        "-4": "Not asked",
        "-5": "Missing"
      }
    },
    "Q275": {
      "description": "Highest educational level: Respondent [ISCED 2011]",
      "question": "What is the highest educational level that you have attained?",
      "topic_tag": null,
      "original_section": "demographics",
      "values": {
        "0": "Early childhood education (ISCED 0) / no education",
        "1": "Primary education (ISCED 1)",
        "2": "Lower secondary education (ISCED 2)",
        "3": "Upper secondary education (ISCED 3)",
        "4": "Post-secondary non-tertiary education (ISCED 4)",
        "5": "Short-cycle tertiary education (ISCED 5)",
        "6": "Bachelor or equivalent (ISCED 6)",
        "7": "Master or equivalent (ISCED 7)",
        "8": "Doctoral or equivalent (ISCED 8)",
        "-1": "Don't know",
        "-2": "No answer",
        "-4": "Not asked",
        "-5": "Missing"
      }
    },
    "Q273": {
      "description": "Marital status",
      "question": "What is your marital status?",
      "topic_tag": null,
      "original_section": "demographics",
      "values": {
        "1": "Married",
        "2": "Living together as married",
        "3": "Divorced",
        "4": "Separated",
        "5": "Widowed",
        "6": "Single",
        "-1": "Don't know",
        "-2": "No answer",
        "-4": "Not asked",
        "-5": "Missing"
      }
    },
    "Q288": {
      "description": "Scale of incomes",
      "question": "What is the level of your household's income counting all wages, salaries, pensions and other incomes on an income scale on which 'lowest' indicates the lowest income group and 'tenth step' the highest income group in your country?",
      "topic_tag": null,
      "original_section": "demographics",
      "values": {
        "1": "Lower step",
        "2": "second step",
        "3": "Third step",
        "4": "Fourth step",
        "5": "Fifth step",
        "6": "Sixth step",
        "7": "Seventh step",
        "8": "Eight step",
        "9": "Nineth step",
        "10": "Tenth step",
        "-1": "Don't know",
        "-2": "No answer",
        "-4": "Not asked",
        "-5": "Missing"
      }
    },
    "Q1": {
      "description": "Important in life: Family",
      "question": "In your life, would you say family is very important, rather important, not very important or not important at all?",
      "topic_tag": "traditionalism",
      "original_section": "identity_religion_values",
      "values": {
        "1": "Very important",
        "2": "Rather important",
        "3": "Not very important",
        "4": "Not at all important",
        "-1": "Don't know",
        "-2": "No answer",
        "-4": "Not asked in this country",
        "-5": "Missing"
      }
    },
    "Q4": {
      "description": "Important in life: Politics",
      "question": "In your life, would you say politics is very important, rather important, not very important or not important at all?",
      "topic_tag": "traditionalism",
      "original_section": "identity_religion_values",
      "values": {
        "1": "Very important",
        "2": "Rather important",
        "3": "Not very important",
        "4": "Not at all important",
        "-1": "Don't know",
        "-2": "No answer",
        "-4": "Not asked in this country",
        "-5": "Missing"
      }
    },
    "Q5": {
      "description": "Important in life: Work",
      "question": "In your life, would you say work is very important, rather important, not very important or not important at all?",
      "topic_tag": "traditionalism",
      "original_section": "identity_religion_values",
      "values": {
        "1": "Very important",
        "2": "Rather important",
        "3": "Not very important",
        "4": "Not at all important",
        "-1": "Don't know",
        "-2": "No answer",
        "-4": "Not asked in this country",
        "-5": "Missing"
      }
    },
    "Q6": {
      "description": "Important in life: Religion",
      "question": "In your life, would you say religion is very important, rather important, not very important or not important at all?",
      "topic_tag": "religious_values",
      "original_section": "identity_religion_values",
      "values": {
        "1": "Very important",
        "2": "Rather important",
        "3": "Not very important",
        "4": "Not at all important",
        "-1": "Don't know",
        "-2": "No answer",
        "-4": "Not asked in this country",
        "-5": "Missing"
      }
    },
    "Q199": {
      "description": "Interest in politics",
      "question": "How interested would you say you are in politics?",
      "topic_tag": "political_interest",
      "original_section": "political_participation",
      "values": {
        "1": "Very interested",
        "2": "Somewhat interested",
        "3": "Not very interested",
        "4": "Not at all interested",
        "-1": "Don't know",
        "-2": "No answer",
        "-4": "Not asked",
        "-5": "Missing"
      }
    },
    "Q240": {
      "description": "Left-right political scale",
      "question": "In politics, people often describe their views as more left-wing or right-wing. Where would you place your own views on a scale from 1 to 10, where 1 means 'left' and 10 means 'right'?",
      "topic_tag": "democratic_values",
      "original_section": "political_attitudes",
      "values": {
        "1": "Far left",
        "2": "Far left",
        "3": "Left",
        "4": "Left",
        "5": "Center",
        "6": "Center",
        "7": "Right",
        "8": "Right",
        "9": "Far right",
        "10": "Far right",
        "-1": "Don't know",
        "-2": "No answer",
        "-4": "Not asked",
        "-5": "Missing"
      }
    },
    "Q46": {
      "description": "Feeling of happiness",
      "question": "Taking all things together, how happy are you?",
      "topic_tag": "life_satisfaction",
      "original_section": "identity_religion_values",
      "values": {
        "1": "Very happy",
        "2": "Quite happy",
        "3": "Not very happy",
        "4": "Not at all happy",
        "-1": "Don't know",
        "-2": "No answer",
        "-4": "Not asked",
        "-5": "Missing"
      }
    }
  },
  "target_questions": {
    "Q64": {
      "description": "Confidence: Churches",
      "question": "How much confidence do you have in the churches?",
      "topic_tag": "institutional_confidence",
      "original_section": "institutional_trust",
      "values": {
        "1": "A great deal",
        "2": "Quite a lot",
        "3": "Not very much",
        "4": "None at all",
        "-1": "Don't know",
        "-2": "No answer",
        "-4": "Not asked",
        "-5": "Missing"
      }
    },
    "Q65": {
      "description": "Confidence: Armed Forces",
      "question": "How much confidence do you have in the armed forces?",
      "topic_tag": "institutional_confidence",
      "original_section": "institutional_trust",
      "values": {
        "1": "A great deal",
        "2": "Quite a lot",
        "3": "Not very much",
        "4": "None at all",
        "-1": "Don't know",
        "-2": "No answer",
        "-4": "Not asked",
        "-5": "Missing"
      }
    },
    "Q69": {
      "description": "Confidence: The Police",
      "question": "How much confidence do you have in the police?",
      "topic_tag": "institutional_confidence",
      "original_section": "institutional_trust",
      "values": {
        "1": "A great deal",
        "2": "Quite a lot",
        "3": "Not very much",
        "4": "None at all",
        "-1": "Don't know",
        "-2": "No answer",
        "-4": "Not asked",
        "-5": "Missing"
      }
    },
    "Q71": {
      "description": "Confidence: The Government",
      "question": "How much confidence do you have in the government?",
      "topic_tag": "institutional_confidence",
      "original_section": "institutional_trust",
      "values": {
        "1": "A great deal",
        "2": "Quite a lot",
        "3": "Not very much",
        "4": "None at all",
        "-1": "Don't know",
        "-2": "No answer",
        "-4": "Not asked",
        "-5": "Missing"
      }
    },
    "Q73": {
      "description": "Confidence: Parliament",
      "question": "How much confidence do you have in parliament?",
      "topic_tag": "institutional_confidence",
      "original_section": "institutional_trust",
      "values": {
        "1": "A great deal",
        "2": "Quite a lot",
        "3": "Not very much",
        "4": "None at all",
        "-1": "Don't know",
        "-2": "No answer",
        "-4": "Not asked",
        "-5": "Missing"
      }
    },
    "Q34": {
      "description": "Jobs scarce: Employers should give priority to (nation) people than immigrants",
      "question": "Do you agree, disagree or neither agree nor disagree with the following statement: 'When jobs are scarce, employers should give priority to people of this country over immigrants.'?",
      "topic_tag": "economic_policy",
      "original_section": "identity_religion_values",
      "values": {
        "1": "Agree strongly",
        "2": "Agree",
        "3": "Neither agree nor disagree",
        "4": "Disagree",
        "5": "Disagree strongly",
        "-1": "Don't know",
        "-2": "No answer",
        "-4": "Not asked",
        "-5": "Missing"
      }
    },
    "Q152": {
      "description": "Aims of country: first choice",
      "question": "What do you think should be the most important goal for this country over the next ten years? Economic growth, strong defence, public participation, or environmental improvement?",
      "topic_tag": "democratic_values",
      "original_section": "identity_religion_values",
      "values": {
        "1": "A high level of economic growth",
        "2": "Strong defence forces",
        "3": "People have more say about how things are done",
        "4": "Trying to make our cities and countryside more beautiful",
        "-1": "Don't know",
        "-2": "No answer",
        "-4": "Not asked",
        "-5": "Missing"
      }
    },
    "Q28": {
      "description": "Pre-school child suffers with working mother",
      "question": "How much do you agree or disagree with the following statement: 'When a mother works for pay, the children suffer'?",
      "topic_tag": "gender_attitudes",
      "original_section": "identity_religion_values",
      "values": {
        "1": "Agree strongly",
        "2": "Agree",
        "3": "Disagree",
        "4": "Strongly disagree",
        "-1": "Don't know",
        "-2": "No answer",
        "-4": "Not asked",
        "-5": "Missing"
      }
    },
    "Q29": {
      "description": "Men make better political leaders than women do",
      "question": "How much do you agree or disagree with the following statement: 'On the whole, men make better political leaders than women do'?",
      "topic_tag": "gender_attitudes",
      "original_section": "identity_religion_values",
      "values": {
        "1": "Agree strongly",
        "2": "Agree",
        "3": "Disagree",
        "4": "Strongly disagree",
        "-1": "Don't know",
        "-2": "No answer",
        "-4": "Not asked",
        "-5": "Missing"
      }
    }
  }
}

profile_features = WVS_METADATA["profile_features"]
target_questions = WVS_METADATA["target_questions"]

print(f"Profile features: {len(profile_features)}")
for code, info in profile_features.items():
    print(f"  {code}: {info['description'][:60]}")
print(f"\nTarget questions: {len(target_questions)}")
for code, info in target_questions.items():
    print(f"  {code}: {info['description'][:60]}")

In [ ]:
#@title Load WVS respondent data
import os

# Option 1: Load from file (instructor will provide this)
WVS_DATA_PATH = "wvs_sample.csv"  # Update this path

if os.path.exists(WVS_DATA_PATH):
    wvs_data = pd.read_csv(WVS_DATA_PATH)
    print(f"Loaded WVS data: {len(wvs_data)} respondents")
else:
    print("WVS data not found. Generating synthetic fallback...")
    # Synthetic fallback: 100 respondents with plausible WVS-like data
    np.random.seed(42)
    n = 100
    wvs_data = pd.DataFrame({
        "Q260": np.random.choice([1, 2], n),                    # sex
        "Q262": np.random.randint(18, 80, n),                   # age
        "Q275": np.random.choice(range(0, 9), n),               # education
        "Q273": np.random.choice([1, 2, 3, 4, 5, 6], n),       # marital
        "Q288": np.random.choice(range(1, 11), n),              # income
        "Q1": np.random.choice([1, 2, 3, 4], n),               # family importance
        "Q4": np.random.choice([1, 2, 3, 4], n),               # politics importance
        "Q5": np.random.choice([1, 2, 3, 4], n),               # work importance
        "Q6": np.random.choice([1, 2, 3, 4], n),               # religion importance
        "Q199": np.random.choice([1, 2, 3, 4], n),             # political interest
        "Q240": np.random.choice(range(1, 11), n),             # left-right
        "Q46": np.random.choice([1, 2, 3, 4], n),              # happiness
        # Targets
        "Q64": np.random.choice([1, 2, 3, 4], n),              # trust churches
        "Q65": np.random.choice([1, 2, 3, 4], n),              # trust armed forces
        "Q69": np.random.choice([1, 2, 3, 4], n),              # trust police
        "Q71": np.random.choice([1, 2, 3, 4], n),              # trust government
        "Q73": np.random.choice([1, 2, 3, 4], n),              # trust parliament
        "Q34": np.random.choice([1, 2, 3, 4, 5], n),           # jobs priority
        "Q152": np.random.choice([1, 2, 3, 4], n),             # country aims
        "Q28": np.random.choice([1, 2, 3, 4], n),              # working mothers
        "Q29": np.random.choice([1, 2, 3, 4], n),              # men better leaders
        "B_COUNTRY": np.random.choice([826, 840, 276, 250], n), # UK, US, DE, FR
    })
    print(f"Generated {n} synthetic respondents (for demonstration)")

print(f"Columns: {list(wvs_data.columns[:10])}...")

In [ ]:
#@title Helper: build profile from respondent data

def get_value_label(code, raw_value, metadata):
    """Convert a raw survey value to its text label."""
    if code not in metadata:
        return str(raw_value)
    values_map = metadata[code].get("values", {})
    raw_str = str(int(raw_value)) if not pd.isna(raw_value) else str(raw_value)
    return values_map.get(raw_str, f"[code {raw_str}]")

def build_profile(respondent, feature_codes, metadata):
    """Build a Q&A profile string from a respondent row and selected features."""
    lines = []
    for code in feature_codes:
        if code not in respondent.index or pd.isna(respondent[code]):
            continue
        info = metadata.get(code, {})
        question = info.get("question", code)
        label = get_value_label(code, respondent[code], metadata)
        if label.startswith("[code") or "Don" in label or label == "No answer":
            continue
        lines.append(f"Q: {question}\nA: {label}")
    return "\n\n".join(lines)

# Demo: build a profile for the first respondent
demo = wvs_data.iloc[0]
demo_profile = build_profile(demo, list(profile_features.keys()), profile_features)
print("Example profile:")
print(demo_profile)

### Exercise 5: Predict a Real Person's Survey Response

Pick a respondent from the WVS data. Build a profile from their demographic features. Then ask the model to predict how they answered a question you hold out.

The key question: **how much information does the model need to predict correctly?** Try with a sparse profile (3 features) and a rich one (8+ features). Which features matter most?

In [ ]:
# Pick a respondent
respondent = wvs_data.iloc[5]  # change this index to try different people

# Choose which features to include in the profile
# Start sparse, then add more. Which features help prediction?
selected_features = ["Q260", "Q262", "Q275"]  # YOUR CHOICE: add or remove

# Choose a target question to predict
target_code = "Q71"  # YOUR CHOICE: pick from target_questions

# Build the profile
profile_text = build_profile(respondent, selected_features, profile_features)
print("PROFILE:")
print(profile_text)

# Get ground truth
target_info = target_questions[target_code]
true_answer = get_value_label(target_code, respondent[target_code], target_questions)
options = [v for k, v in target_info["values"].items() if not k.startswith("-")]

# Construct the prediction prompt
prompt = (
    "Below is a profile of a real survey respondent. Based on this profile, "
    "predict how they would answer the following question.\n\n"
    f"RESPONDENT PROFILE:\n{profile_text}\n\n"
    f"QUESTION: {target_info['question']}\n"
    f"OPTIONS: {\", \".join(options)}\n\n"
    "Answer with ONLY the option text, nothing else."
)

prediction = generate_response(prompt, max_new_tokens=30)
print(f"\nTARGET: {target_info['question']}")
print(f"OPTIONS: {options}")
print(f"MODEL PREDICTION: {prediction}")
print(f"TRUE ANSWER: {true_answer}")
print(f"CORRECT: {prediction.strip().lower() in true_answer.lower() or true_answer.lower() in prediction.strip().lower()}")

### Exercise 6: What Matters for Prediction?

Run the same target question across multiple respondents and profile configurations. This is the core research question: which features does the model use, and does more information always help?

Compare:
- **Sparse profile** (demographics only: sex, age, education)
- **Rich profile** (demographics + values + political orientation)
- **Targeted profile** (only features you think are relevant to the target question)

In [ ]:
# Compare prediction accuracy across profile richness levels
sparse_features = ["Q260", "Q262", "Q275"]           # demographics only
rich_features = list(profile_features.keys())         # everything
targeted_features = ["Q240", "Q199", "Q4"]            # YOUR CHOICE: what do you think matters?

target_code = "Q71"  # trust in government
target_info = target_questions[target_code]
options = [v for k, v in target_info["values"].items() if not k.startswith("-")]

sample = wvs_data.sample(10, random_state=42)

for label, features in [("sparse", sparse_features), ("rich", rich_features), ("targeted", targeted_features)]:
    correct = 0
    for _, resp in sample.iterrows():
        profile_text = build_profile(resp, features, profile_features)
        true_answer = get_value_label(target_code, resp[target_code], target_questions)
        if true_answer.startswith("[code") or "Don" in true_answer:
            continue

        prompt = (
            "Below is a profile of a real survey respondent. Predict their answer.\n\n"
            f"PROFILE:\n{profile_text}\n\n"
            f"QUESTION: {target_info['question']}\n"
            f"OPTIONS: {\", \".join(options)}\n"
            "Answer with ONLY the option text."
        )
        pred = generate_response(prompt, max_new_tokens=30).strip()
        if true_answer.lower() in pred.lower() or pred.lower() in true_answer.lower():
            correct += 1

    print(f"{label:10s} ({len(features)} features): {correct}/10 correct")

### What this tells us

Silicon sampling can reproduce aggregate patterns for well-studied topics (party identification predicts policy views). But it struggles with:

**WEIRD bias:** LLMs are trained primarily on English-language, Western, educated, internet-connected text. Simulating respondents outside this distribution is unreliable.

**Sycophancy:** Models tend to agree with the framing of the question, inflating agreement rates.

**Flat distributions:** For niche or controversial topics, LLMs produce uncommitted, centrist responses rather than the polarized distributions observed in real data.

**Refusal:** Safety training causes models to refuse questions about sensitive topics that real respondents would answer.

The emerging consensus: simulated responses are most useful for **pilot testing** (identifying confusing survey items), **hypothesis generation** (exploring which variables might matter), and **benchmarking** models against known human distributions. They are not replacements for real human data.

The `synthetic_sampling` package (Oxford LLMs Research) provides tools for running these experiments at scale across multiple surveys (WVS, ESS, Afrobarometer, and others), with built-in robustness tests for surface form sensitivity, option ordering effects, and feature order invariance.

---

## What you learned today

1. **Information extraction** turns unstructured text into structured data, but requires faithfulness evaluation. The model can hallucinate, omit, or distort.

2. **RAG** grounds the model's outputs in your actual data. The pipeline (chunk, embed, index, retrieve, generate) is the standard approach for working with large corpora.

3. **Simulated agents** can reproduce some human response patterns, but profile richness matters, and certain biases (WEIRD, sycophancy, flat distributions) limit their validity as substitutes for real data.

## Bridge to Day 5

Over four days you have built a toolkit: embeddings, prompting, validation, fine-tuning, extraction, RAG, simulation. Tomorrow you learn how to **connect these pieces into autonomous workflows**. The RAG pipeline becomes a tool an agent can call. The extraction prompts become functions in a planning loop.

---

Course website: [llmsforsocialscience.net](https://llmsforsocialscience.net)

---
# Extension Exercises
*For students who finish early or want to go deeper.*

---

### Extension A: Metadata-Filtered Retrieval

The cross-party comparison in Exercise 4 may have retrieved text from the wrong party. Metadata filtering solves this by restricting the search to a specific party, year, or topic before doing semantic search.

In [ ]:
#@title Stretch: Metadata-filtered retrieval

def retrieve_filtered(query, k=3, party=None, year=None, topic=None):
    """Retrieve with optional metadata filters applied before semantic search."""
    # Filter the corpus first
    mask = pd.Series([True] * len(corpus_df))
    if party:
        mask &= corpus_df['party'] == party
    if year:
        mask &= corpus_df['year'] == year
    if topic:
        mask &= corpus_df['topic'] == topic

    filtered_indices = corpus_df[mask].index.tolist()
    if not filtered_indices:
        return []

    # Get embeddings for filtered chunks only
    filtered_embeddings = chunk_embeddings[filtered_indices]

    # Build a temporary index
    temp_index = faiss.IndexFlatIP(embedding_dim)
    temp_index.add(filtered_embeddings)

    # Search
    query_vec = embed_model.encode([query], convert_to_numpy=True)
    faiss.normalize_L2(query_vec)
    scores, local_indices = temp_index.search(query_vec, min(k, len(filtered_indices)))

    results = []
    for score, local_idx in zip(scores[0], local_indices[0]):
        global_idx = filtered_indices[local_idx]
        row = corpus_df.iloc[global_idx]
        results.append({
            'text': row['text'], 'party': row['party'],
            'year': row['year'], 'topic': row['topic'],
            'score': float(score), 'id': row['id'],
        })
    return results


# Example: compare Labour's position on the economy across elections
print("Labour's economic position over time:")
print("=" * 60)
for year in [2017, 2019, 2024]:
    results = retrieve_filtered("economic policy and spending", k=1, party="Labour", year=year)
    if results:
        r = results[0]
        print(f"\n{year}: {r['text'][:200]}")
    else:
        print(f"\n{year}: No matching excerpts found")


---
# Solutions

---

In [ ]:
#@title Solution: Exercise 1 (Extract and Verify)
extraction_prompt = (
    "Read the following excerpt from a UK party manifesto and extract structured information.\n\n"
    "Text: {text}\n\n"
    "Respond with ONLY a JSON object in this exact format:\n"
    '{{"policy": "what is being proposed (one sentence)",'
    ' "position": "ideological direction (e.g. increase spending, cut taxes)",'
    ' "target_group": "who benefits or is affected",'
    ' "justification": "why the party says this is needed"}}'
)

test_excerpts = corpus_df.sample(5, random_state=42)
for _, row in test_excerpts.iterrows():
    print(f"[{row['party']} {row['year']}, {row['topic']}]")
    response = generate_response(extraction_prompt.format(text=row['text']), max_new_tokens=200)
    try:
        cleaned = response.strip().strip('`')
        if cleaned.startswith('json'): cleaned = cleaned[4:].strip()
        parsed = json.loads(cleaned)
        for k, v in parsed.items():
            print(f"  {k}: {v}")
    except json.JSONDecodeError:
        print(f"  [Raw]: {response[:200]}")
    print()

---
*This notebook is part of the [LLMs for Social Science](https://llmsforsocialscience.net) course.*